---

# SystemVerilog FPGA Development with the OSS CAD Suite

Welcome! This handout covers how to install, set up, and use the **OSS CAD Suite** (maintained by [YosysHQ](https://github.com/YosysHQ/oss-cad-suite-build)) on **Windows, macOS, and Linux**. 

The OSS CAD Suite is a self-contained, pre-compiled bundle containing almost every open-source tool you need for digital design: synthesis, formal verification, place-and-route, simulation, and hardware programming.

---

## 1. Installation & Environment Setup

The OSS CAD Suite is portable. You do not need admin/root privileges to install it—you simply extract it and source its environment.

### Step 1: Download
Go to the official release page:  
👉 **[YosysHQ OSS CAD Suite Releases](https://github.com/YosysHQ/oss-cad-suite-build/releases/latest)**

Download the archive matching your operating system:
* **Linux (x86_64):** `oss-cad-suite-linux-x64-YYYYMMDD.tgz`
* **macOS (Apple Silicon / M-series):** `oss-cad-suite-darwin-arm64-YYYYMMDD.tgz`
* **macOS (Intel):** `oss-cad-suite-darwin-x64-YYYYMMDD.tgz`
* **Windows (64-bit):** `oss-cad-suite-windows-x64-YYYYMMDD.exe` (self-extracting 7-Zip)

---

### Step 2: Extraction & Setup by OS

#### 🐧 Linux
Extract the archive into your home directory or tools folder:
```bash
tar -xvzf oss-cad-suite-linux-x64-*.tgz -C ~/
```
To activate the tools in your current terminal:
```bash
source ~/oss-cad-suite/environment
```
*(Optional)* Add `source ~/oss-cad-suite/environment` to the bottom of your `~/.bashrc` or `~/.zshrc` to have the tools available in every terminal.

---

#### 🍏 macOS
Extract the archive:
```bash
tar -xvzf oss-cad-suite-darwin-*.tgz -C ~/
```
To activate:
```bash
source ~/oss-cad-suite/environment
```

> **macOS Security Note (Gatekeeper):**  
> macOS may block downloaded binaries from running. If you get a warning that a binary "cannot be opened because it is from an unidentified developer," clear the quarantine attribute by running:
> ```bash
> xattr -r -d com.apple.quarantine ~/oss-cad-suite
> ```

---

#### 🪟 Windows
1. Double-click the downloaded `.exe` to extract it (e.g., into `C:\oss-cad-suite`).
2. To use the suite, simply run **`start.bat`** located inside `C:\oss-cad-suite\`. This opens a command prompt with all paths and Python environments already configured.
3. Alternatively, in PowerShell:
   ```powershell
   & "C:\oss-cad-suite\environment.bat"
   ```

> **Windows USB Driver Note:**  
> To program FPGAs over USB via `openFPGALoader`, Windows may require WinUSB / libusb drivers instead of default FTDI COM drivers. If your board is not detected, use [Zadig](https://zadig.akeo.ie/) to assign the **WinUSB** driver to Interface 0 of your FPGA board.

---

### Step 3: Verify the Installation (Interactive Cell)
Run the cell below in your notebook to confirm that your environment is active and the tools are visible.

```bash
# Check that key tools are installed and in PATH
yosys --version
verilator --version
openFPGALoader --version
```

---

## 2. Core Tools in the Suite: What They Do & How to Use Them

When doing SystemVerilog FPGA design, you will transition through three main phases: **Linting & Simulation**, **Synthesis & Implementation**, and **Hardware Programming**.

Here are the primary programs you will use:

```
+-------------------------------------------------------------------+
|               1. Simulation & Verification                        |
|                                                                   |
|   [ SystemVerilog Source ] ---> Verilator / Icarus Verilog        |
|                                       |                           |
|                                       v                           |
|                            GTKWave / Surfer (Waveforms)           |
+-------------------------------------------------------------------+
                                       |
                                       v
+-------------------------------------------------------------------+
|               2. Synthesis & Place-and-Route                      |
|                                                                   |
|   [ SystemVerilog Source ] ---> Yosys (Synthesis to Gate-Level)   |
|                                       |                           |
|                                       v                           |
|                                 nextpnr (Place & Route)           |
|                                       |                           |
|                                       v                           |
|                         Packer (icepack / ecppack / gowin_pack)   |
+-------------------------------------------------------------------+
                                       |
                                       v
+-------------------------------------------------------------------+
|               3. Deployment to Hardware                           |
|                                                                   |
|   [ Bitstream (.bin/.fs) ] ---> openFPGALoader ---> [ FPGA Board ]|
+-------------------------------------------------------------------+
```

---

### 1. Verilator
* **What it does:** Verilator is a high-performance **SystemVerilog simulator and linter**. Instead of interpreting Verilog directly, it translates SystemVerilog into optimized C++ models and compiles them into a native executable testbench.
* **Why we use it:** It has the best SystemVerilog language support among open-source tools and catches bugs, unassigned wires, and race conditions very early via linting.
* **Basic Usage:**
  * **Linting only:**
    ```bash
    verilator --lint-only -Wall top.sv
    ```
  * **Simulation with C++ testbench:**
    ```bash
    verilator -Wall --cc top.sv --exe tb_top.cpp --trace
    make -C obj_dir -f Vtop.mk
    ./obj_dir/Vtop
    ```

---

### 2. Yosys
* **What it does:** Yosys is the **RTL Synthesis engine**. It parses your hardware description (Verilog/SystemVerilog), elaborates the hierarchy, optimizes logic, and maps high-level constructs (ALUs, multiplexers, state machines) to hardware primitives like LUTs (Look-Up Tables), Flip-Flops, and BRAMs.
* **Why we use it:** It produces a gate-level netlist (typically formatted as JSON) that the Place-and-Route engine understands.
* **Basic SystemVerilog Usage:**
  * To tell Yosys to parse SystemVerilog constructs, pass the `-sv` flag:
    ```bash
    # Synthesizing for a Lattice iCE40 FPGA
    yosys -p "read_verilog -sv top.sv; synth_ice40 -top top -json top.json"
    
    # Synthesizing for a Lattice ECP5 FPGA
    yosys -p "read_verilog -sv top.sv; synth_ecp5 -top top -json top.json"
    ```

---

### 3. nextpnr
* **What it does:** **Place-and-Route (P&R)**. Once Yosys produces the logical netlist, `nextpnr` takes that netlist along with a pin constraint file (e.g., `.pcf` or `.lpf`) and:
  1. Decides *where* each LUT and flip-flop will physically sit on the FPGA silicon (**Placement**).
  2. Connects all wires through the chip's internal routing matrix (**Routing**).
  3. Checks that your design meets clock timing constraints (**Timing Analysis**).
* **Target Variants:** `nextpnr-ice40` (Lattice iCE40), `nextpnr-ecp5` (Lattice ECP5), `nextpnr-gowin` (Gowin LittleBee/Arora), and `nextpnr-nexus`.
* **Basic Usage:**
  ```bash
  nextpnr-ice40 --hx8k --package ct256 --pcf pins.pcf --json top.json --asc top.asc
  ```

---

### 4. Bitstream Packers (`icepack`, `ecppack`, `gowin_pack`)
* **What it does:** Converts the physical placement ASCII text file generated by `nextpnr` into the proprietary **binary bitstream format** that the FPGA hardware actually reads on power-up.
* **Basic Usage:**
  * For iCE40: `icepack top.asc top.bin`
  * For ECP5: `ecppack top.config top.bit`

---

### 5. openFPGALoader
* **What it does:** A universal, open-source utility for **flashing and programming FPGAs**.
* **Why we use it:** It works across almost all vendors (Lattice, Gowin, Xilinx, Altera) and programmers (FTDI, JTAG, CMSIS-DAP) with a single command.
* **Basic Usage:**
  ```bash
  # Detect attached FPGA boards
  openFPGALoader --detect

  # Program SRAM (temporary, resets on power cycle)
  openFPGALoader -b ice40_generic top.bin

  # Write directly to SPI Flash (permanent across reboots)
  openFPGALoader -b ice40_generic -f top.bin
  ```

---

### 6. Waveform Viewers: GTKWave & Surfer
* **What they do:** Graphical tools used to inspect the `.vcd` (Value Change Dump) or `.fst` files generated during simulation.
* **Why we use them:** They let you view clocks, data busses, state machine transitions, and glitch artifacts over time.
* **Basic Usage:**
  ```bash
  gtkwave dump.vcd
  # or
  surfer dump.vcd
  ```

---

## 3. Hands-On Walkthrough: SystemVerilog Blinky

Let's test the workflow end-to-end directly from this notebook.

### Step 3.1: Create a SystemVerilog Design

```python
%%writefile blinky.sv
// Simple 32-bit counter blinker written in SystemVerilog
module blinky (
    input  logic clk,
    output logic led
);

    logic [24:0] counter = 25'd0;

    always_ff @(posedge clk) begin
        counter <= counter + 1'b1;
    end

    // Assign the MSB to the LED so it toggles at visible human speeds
    assign led = counter[24];

endmodule
```

---

### Step 3.2: Lint with Verilator

Before synthesizing, run a lint check on your SystemVerilog file to catch potential errors:

```python
!verilator --lint-only -Wall blinky.sv
```

*(If this produces no errors, your code passes open-source SV linting standards!)*

---

### Step 3.3: Synthesize with Yosys

Run Yosys in batch mode using the `-p` parameter. Here, we target a generic Lattice iCE40 architecture:

```python
!yosys -p "read_verilog -sv blinky.sv; synth_ice40 -top blinky -json blinky.json"
```

---

### Step 3.4: Place and Route with nextpnr

Next, run `nextpnr` to map the synthesized JSON to a target package (e.g., an iCE40 HX1K in a TQ144 package):

```python
# Note: In a real project you would supply a pin constraint file using --pcf pins.pcf
!nextpnr-ice40 --hx1k --package tq144 --json blinky.json --asc blinky.asc
```

---

### Step 3.5: Pack into a Bitstream

```python
!icepack blinky.asc blinky.bin
print("Success! Bitstream 'blinky.bin' is ready to flash.")
```

---

### Step 3.6: Flash with openFPGALoader

Connect your board via USB and run detection:

```python
!openFPGALoader --detect
```

*(When ready to flash, replace `<board_name>` with your board identifier, e.g., `ice40_generic` or `tangnano9k`):*
```python
# !openFPGALoader -b ice40_generic blinky.bin
```

---

## 4. Quick Reference Cheat Sheet

| Task | Command |
| :--- | :--- |
| **Lint SV Code** | `verilator --lint-only -Wall <file.sv>` |
| **Compile & Run (Icarus)** | `iverilog -g2012 -o sim.out <tb.sv> <dut.sv> && vvp sim.out` |
| **Synthesize (Yosys)** | `yosys -p "read_verilog -sv <file.sv>; synth_<arch> -top <top> -json <out.json>"` |
| **Place & Route (nextpnr)** | `nextpnr-<arch> --json <out.json> --<chip_variant> --pcf <pins.pcf> --asc <out.asc>` |
| **View Waveforms** | `gtkwave <trace.vcd>` or `surfer <trace.vcd>` |
| **Detect Hardware** | `openFPGALoader --detect` |
| **Flash Hardware (SRAM)** | `openFPGALoader -b <board_name> <bitstream.bin>` |
| **Flash Hardware (Flash)** | `openFPGALoader -b <board_name> -f <bitstream.bin>` |